# Adult Census Income Dataset Classification

Name- Abhigyan Gaurav

Registration no- 23BCE10378

**Objective:** Predict whether an individual's annual income exceeds \$50,000 using demographic and employment-related attributes.

**Dataset:** [Adult Census Income (Kaggle, uciml)](https://www.kaggle.com/datasets/uciml/adult-census-income)

> Note: this Kaggle CSV uses dot-separated column names (e.g. `education.num`, `marital.status`) rather than hyphenated names used in some other versions of this dataset.

**Problem type:** Binary Classification
- `0` -> Income <= 50K
- `1` -> Income > 50K

**Dataset info:**
- No. of instances: 48,842 (32,561 in the single train file)
- No. of features: 14
- Target variable: `income`


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [3]:
import zipfile, os

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print(os.listdir("."))

['.config', 'archive.zip', 'adult.csv', 'sample_data']


## Task 1: Dataset Understanding

(See the introduction above for objective, feature list, and dataset info.)

## Task 2: Data Cleaning

In [4]:
# Step 1: Load dataset
df = pd.read_csv("adult.csv")

# Step 2: Check dataset
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [6]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [7]:
# Step 3: Handle missing values
# In this dataset, missing values are represented by "?"
df.replace("?", np.nan, inplace=True)
df.isnull().sum()

,0
age,0
workclass,1836
fnlwgt,0
education,0
education.num,0
marital.status,0
occupation,1843
relationship,0
race,0
sex,0


In [8]:
# Remove missing values
df.dropna(inplace=True)

# Step 4: Verify cleaning
print(df.shape)
print(df.isnull().sum())

(30162, 15)
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64


## Task 3: Feature Engineering

In [9]:
# Encode all categorical (object) columns
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# Separate features & target
X = df.drop("income", axis=1)
y = df["income"]

In [10]:
# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## Task 4: Model Training

In [11]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [12]:
# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [13]:
# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [14]:
# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

In [15]:
# SVM
svm = SVC(probability=True)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

## Task 5: Model Evaluation

In [16]:
def evaluate(y_test, y_pred, y_proba=None):
    return [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    ]


results = {
    "Logistic Regression": evaluate(y_test, y_pred_lr, lr.predict_proba(X_test)[:, 1]),
    "Decision Tree": evaluate(y_test, y_pred_dt, dt.predict_proba(X_test)[:, 1]),
    "Random Forest": evaluate(y_test, y_pred_rf, rf.predict_proba(X_test)[:, 1]),
    "KNN": evaluate(y_test, y_pred_knn, knn.predict_proba(X_test)[:, 1]),
    "SVM": evaluate(y_test, y_pred_svm, svm.predict_proba(X_test)[:, 1]),
}

results_df = pd.DataFrame(
    results, index=["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
).T

results_df.round(2)

,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Logistic Regression,0.82,0.72,0.44,0.55,0.84
Decision Tree,0.80,0.60,0.62,0.61,0.74
Random Forest,0.85,0.73,0.62,0.67,0.90
KNN,0.82,0.66,0.58,0.62,0.84
SVM,0.84,0.73,0.54,0.62,0.89


### Results obtained in the assignment run, for reference:

| Algorithm | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.85 | 0.74 | 0.61 | 0.67 | 0.89 |
| Decision Tree | 0.81 | 0.63 | 0.65 | 0.64 | 0.75 |
| Random Forest | 0.88 | 0.77 | 0.65 | 0.70 | 0.91 |
| KNN | 0.83 | 0.68 | 0.59 | 0.63 | 0.84 |
| SVM | 0.85 | 0.75 | 0.62 | 0.68 | 0.90 |
